# Switch to 'exploration' schema

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS exploration;

USE CATALOG `operations-catalog`;
USE SCHEMA exploration;

SELECT current_schema() AS schema, current_catalog() AS catalog;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW bronze_purchase_order AS
SELECT *
FROM `operations-catalog`.supplychain.bronze_purchase_order;

In [0]:
%sql
SELECT * FROM bronze_purchase_order
LIMIT 5;

In [0]:

%sql
CREATE OR REPLACE TEMP VIEW bronze_purchase_order_parsed AS
SELECT
    TRY_CAST(key AS STRING) AS key,
    TRY_CAST(value AS STRING) AS value,
    *
    EXCEPT (key, value)
FROM bronze_purchase_order;

In [0]:
%sql
SELECT *
FROM bronze_purchase_order_parsed
LIMIT 5;

In [0]:
%sql
DESCRIBE bronze_purchase_order_parsed;

In [0]:
from pyspark.sql.functions import from_json, parse_json
from pyspark.sql.types import *

key_schema = StructType([
    StructField(
        "schema",
        StructType([
            StructField("type", StringType()),
            StructField(
                "fields",
                ArrayType(
                    StructType([
                        StructField("type", StringType()),
                        StructField("optional", BooleanType()),
                        StructField("field", StringType())
                    ])
                )
            ),
            StructField("optional", BooleanType()),
            StructField("name", StringType())
        ])
    ),
    StructField(
        "payload",
        StructType([
            StructField("PO_ID", StringType())
        ])
    )
])

bronze_purchase_order_parsed = spark.table("bronze_purchase_order_parsed")
bronze_purchase_order_casted = bronze_purchase_order_parsed.withColumn(
    "key",
    from_json("key", key_schema)
).withColumn(
    "value",
    parse_json("value")
)

bronze_purchase_order_casted.show(5)

In [0]:
bronze_purchase_order_casted.printSchema()

In [0]:
bronze_purchase_order_parsed.printSchema()

In [0]:
bronze_purchase_order_casted.createOrReplaceTempView("bronze_purchase_order_casted")

In [0]:
%sql
CREATE OR REPLACE TABLE bronze_purchase_order_flat AS
SELECT 
key.payload.PO_ID as PO_ID,
TRY_CAST(value:payload:SupplierID AS STRING) as SupplierID,
value:payload:OrderDate as OrderDate,
value:payload:ExpectedDeliveryDate as ExpectedDeliveryDate,
TRY_CAST(value:payload:Status AS STRING) as Status,
value:payload:TotalCost as TotalCost,
TRY_CAST(value:payload:__op AS STRING) as Operation,
value:payload:__deleted as Deleted,
*
EXCEPT (key, value)
FROM bronze_purchase_order_casted;

In [0]:
%sql DESCRIBE bronze_purchase_order_flat

In [0]:
%sql
SELECT DISTINCT Status FROM bronze_purchase_order_flat;

In [0]:
%sql
select distinct Operation from bronze_purchase_order_flat;

In [0]:
%sql
select * from bronze_purchase_order_flat where Status is null limit 5;